# NASA POWER — Solar + Meteorology for Agriculture

**What it is.** NASA's **P**rediction **O**f **W**orldwide **E**nergy **R**esources —
satellite-derived daily solar radiation and meteorology, with an **AG** community profile.

| | |
|---|---|
| **Coverage** | Global (~0.5°), point API |
| **History** | Daily 1981 → present; hourly/monthly also |
| **Cost / key** | **Free · no key** |
| **Docs** | https://power.larc.nasa.gov/docs/services/api/temporal/daily/ |

**Why it's in Tracera.** Best free source for **solar radiation** (drives ET, biomass, drying)
plus temperature/precip/humidity/wind — global, no key, long record.

In [1]:
# Tracera shared setup — credentials + the ONE sample farm location
# (Every source is queried at this same spot so results are comparable.)
import sys, pathlib, requests, pandas as pd
sys.path.insert(0, str(pathlib.Path.cwd()))
from data_sources._common import SAMPLE_FARM, get_key, field_polygon

LAT, LON = SAMPLE_FARM["lat"], SAMPLE_FARM["lon"]
FIPS, STATE, COUNTY = SAMPLE_FARM["fips"], SAMPLE_FARM["state_alpha"], SAMPLE_FARM["county_name"]
print(SAMPLE_FARM["name"], f"| lat={LAT}, lon={LON} | FIPS {FIPS}")

Story County, Iowa (sample farm) | lat=42.05, lon=-93.5 | FIPS 19169


### 1. Connection test — daily AG parameters at the field

In [2]:
POWER = "https://power.larc.nasa.gov/api/temporal/daily/point"
PARAMS = ["T2M", "T2M_MAX", "T2M_MIN", "PRECTOTCORR", "ALLSKY_SFC_SW_DWN", "RH2M", "WS2M"]

def nasa_power_daily(start, end, lat=LAT, lon=LON, params=PARAMS):
    """start/end as YYYYMMDD. Returns a daily DataFrame (fill value -999 -> NaN)."""
    r = requests.get(POWER, params={"parameters": ",".join(params), "community": "AG",
        "latitude": lat, "longitude": lon, "start": start, "end": end, "format": "JSON"}, timeout=90)
    r.raise_for_status()
    p = r.json()["properties"]["parameter"]
    df = pd.DataFrame(p)
    df.index = pd.to_datetime(df.index, format="%Y%m%d")
    return df.replace(-999.0, pd.NA)

test = nasa_power_daily("20230601", "20230605")
assert not test.empty
test

,T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,ALLSKY_SFC_SW_DWN,RH2M,WS2M
2023-06-01,21.34,24.11,19.18,15.26,13.51,92.17,2.07
2023-06-02,23.75,29.44,18.65,1.16,21.90,76.66,1.57
2023-06-03,24.66,31.21,19.35,0.42,22.85,65.90,2.32
2023-06-04,25.01,31.22,18.56,0.56,24.23,61.39,2.15
2023-06-05,23.42,28.87,16.46,0.37,24.29,66.35,1.33


### 2. Core query — a growing season with solar radiation & GDD

In [3]:
wx = nasa_power_daily("20230501", "20230930").astype(float)
tmax = wx["T2M_MAX"].clip(upper=30)
tmin = wx["T2M_MIN"].clip(lower=10)
wx["GDD_corn"] = (((tmax + tmin) / 2) - 10).clip(lower=0)
print(f"Season — precip {wx['PRECTOTCORR'].sum():.0f} mm | "
      f"mean solar {wx['ALLSKY_SFC_SW_DWN'].mean():.1f} kWh/m2/day | "
      f"GDD(corn) {wx['GDD_corn'].sum():.0f}")
wx[["T2M_MAX", "T2M_MIN", "PRECTOTCORR", "ALLSKY_SFC_SW_DWN", "RH2M", "GDD_corn"]].head()

Season — precip 333 mm | mean solar 20.6 kWh/m2/day | GDD(corn) 1812


,T2M_MAX,T2M_MIN,PRECTOTCORR,ALLSKY_SFC_SW_DWN,RH2M,GDD_corn
2023-05-01,15.42,2.27,0.0,28.68,53.44,2.710
2023-05-02,17.11,2.17,0.0,29.62,48.69,3.555
2023-05-03,21.02,1.36,0.0,28.12,53.31,5.510
2023-05-04,25.92,6.02,0.0,27.32,53.25,7.960
2023-05-05,20.14,8.51,16.2,13.64,86.23,5.070


### Notes & how to extend
- **Parameter catalog:** `ALLSKY_SFC_SW_DWN` (solar), `T2M*` (air temp), `PRECTOTCORR`
  (bias-corrected precip), `RH2M` (humidity), `WS2M`/`WS10M` (wind), `ALLSKY_SFC_PAR_TOT`
  (photosynthetically active radiation). Full list in the docs.
- **Solar radiation** is POWER's standout free variable — most station networks lack it.
- **Regional grids:** the `/regional` endpoint returns a lat/lon box; `/climatology` gives
  long-term normals.